# ActionFlow: Optical Flow vs RGB Action Recognition on KTH

This notebook is the primary deliverable for the ActionFlow project. It runs two end-to-end experiments on the [KTH Actions dataset](https://www.csc.kth.se/cvap/actions/):

1. Optical Flow model - a ResNet-18 trained on stacked dense Farneback flow fields (20-channel input: 10 frame pairs x 2 displacement axes).
2. RGB baseline model - a ResNet-18 trained on center RGB frames.

The notebook is cache-aware and CPU-friendly by default. On a GPU machine it expands to the full dataset automatically; on CPU it prepares a smaller per-class subset so the whole pipeline can finish in a reasonable time.

In [ ]:
%matplotlib inline

import json
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("src").resolve()))

from actionflow.data.dataset import (
    FlowClipDataset,
    RGBClipDataset,
    TEST_PERSONS,
    TRAIN_PERSONS,
    extract_person_id,
    get_train_test_split,
)
from actionflow.data.flow import compute_video_flow, visualize_flow
from actionflow.data.frames import extract_video_frames
from actionflow.models.resnet_flow import build_resnet18_flow
from actionflow.training.metrics import (
    classification_report,
    compute_accuracy,
    compute_confusion_matrix,
    plot_confusion_matrix,
    plot_training_curves,
)
from actionflow.training.trainer import Trainer
from actionflow.utils.seed import seed_everything

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Configuration

The config dataclass controls the notebook pipeline.

- Device: auto-detected - CUDA if available, otherwise CPU.
- CPU defaults: smaller image size, smaller batch size, fewer epochs, and a per-class subset so the notebook remains practical to execute locally.
- GPU defaults: full dataset, larger batch size, and pretrained backbones.
- Set `train_videos_per_class` and `test_videos_per_class` to `None` if you want the full KTH split on CPU too.

In [ ]:
USE_GPU = torch.cuda.is_available()


@dataclass
class ActionFlowConfig:
    data_root: Path = Path("data/kth")
    raw_root: Path = Path("data/kth/raw")
    class_names: tuple[str, ...] = (
        "boxing",
        "handclapping",
        "handwaving",
        "jogging",
        "running",
        "walking",
    )
    resize: tuple[int, int] = (224, 224) if USE_GPU else (112, 112)
    clip_length: int = 10
    frame_stride: int = 2

    # CPU-friendly defaults. Set these to None to use the full dataset.
    train_videos_per_class: int | None = None if USE_GPU else 3
    test_videos_per_class: int | None = None if USE_GPU else 2

    mode: str = "flow"
    pretrained_backbone: bool = USE_GPU
    batch_size: int = 16 if USE_GPU else 4
    epochs: int = 15 if USE_GPU else 2
    lr: float = 1e-3
    weight_decay: float = 1e-4
    scheduler: str = "cosine"
    device: str = "cuda" if USE_GPU else "cpu"
    num_workers: int = 2 if USE_GPU else 0
    seed: int = 42
    output_dir: Path = Path("outputs/flow")

    def __post_init__(self) -> None:
        if self.mode not in {"flow", "rgb"}:
            raise ValueError("mode must be 'flow' or 'rgb'.")
        self.num_classes = len(self.class_names)
        self.input_channels = self.clip_length * 2 if self.mode == "flow" else 3


config = ActionFlowConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)

display(
    pd.DataFrame(
        {
            "value": {
                "device": config.device,
                "mode": config.mode,
                "resize": f"{config.resize[0]}x{config.resize[1]}",
                "clip_length": config.clip_length,
                "frame_stride": config.frame_stride,
                "input_channels": config.input_channels,
                "pretrained_backbone": config.pretrained_backbone,
                "batch_size": config.batch_size,
                "epochs": config.epochs,
                "lr": config.lr,
                "scheduler": config.scheduler,
                "num_workers": config.num_workers,
                "train_videos_per_class": str(config.train_videos_per_class or "all"),
                "test_videos_per_class": str(config.test_videos_per_class or "all"),
            }
        }
    )
)

## 2. Pipeline Helpers

This cell keeps the notebook-specific glue while reusing the canonical code under `src/actionflow/data`:

- Download check: only downloads a class if its `.avi` files are missing.
- Notebook subset selection: uses the official KTH person split to choose a manageable CPU subset.
- Frame extraction: reuses `extract_video_frames`.
- Optical flow: reuses `compute_video_flow`.
- Dataset classes: reuses `FlowClipDataset` and `RGBClipDataset`.
- Inline evaluation: keeps the notebook-friendly plots and tables.

In [ ]:
NOTEBOOK_SAMPLE_CLASSES = ("boxing", "running", "handwaving")
NOTEBOOK_SAMPLE_FLOW_CLASSES = ("boxing", "walking")


def ensure_kth_downloaded(config: ActionFlowConfig) -> Path:
    import urllib.request
    import zipfile

    base_url = "https://www.csc.kth.se/cvap/actions"
    raw_root = config.raw_root
    raw_root.mkdir(parents=True, exist_ok=True)

    for action in config.class_names:
        avi_dir = raw_root / action
        if avi_dir.exists() and list(avi_dir.glob("*.avi")):
            print(f"[SKIP] {action} already downloaded")
            continue
        avi_dir.mkdir(parents=True, exist_ok=True)
        zip_path = raw_root / f"{action}.zip"
        url = f"{base_url}/{action}.zip"
        print(f"[DOWNLOADING] {action}.zip ...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"[EXTRACTING] {action}.zip")
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            zip_file.extractall(avi_dir)
        zip_path.unlink()

    total_videos = sum(len(list((raw_root / class_name).glob("*.avi"))) for class_name in config.class_names)
    if total_videos <= 0:
        raise FileNotFoundError(f"No AVI files found under {raw_root}")
    print(f"Download complete. {total_videos} videos available.")
    return raw_root


def summarize_raw_videos(raw_root: Path, class_names: tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for class_name in class_names:
        avis = list((raw_root / class_name).glob("*.avi"))
        rows.append({"class": class_name, "avi_files": len(avis)})
    return pd.DataFrame(rows)


def select_videos_for_notebook(
    raw_root: Path,
    class_names: tuple[str, ...],
    train_per_class: int | None,
    test_per_class: int | None,
) -> tuple[list[Path], pd.DataFrame]:
    selected: list[Path] = []
    rows: list[dict[str, int | str]] = []

    for class_name in class_names:
        avi_paths = sorted((raw_root / class_name).glob("*.avi"))
        train_videos = [path for path in avi_paths if extract_person_id(path.stem) in TRAIN_PERSONS]
        test_videos = [path for path in avi_paths if extract_person_id(path.stem) in TEST_PERSONS]
        if train_per_class is not None:
            train_videos = train_videos[:train_per_class]
        if test_per_class is not None:
            test_videos = test_videos[:test_per_class]

        selected.extend(train_videos)
        selected.extend(test_videos)
        rows.append(
            {
                "class": class_name,
                "train_videos": len(train_videos),
                "test_videos": len(test_videos),
                "total_selected": len(train_videos) + len(test_videos),
            }
        )

    return selected, pd.DataFrame(rows)


def extract_frames_for_selected(
    avi_paths: list[Path],
    raw_root: Path,
    frames_root: Path,
    resize: tuple[int, int],
) -> int:
    total_created = 0
    for avi_path in tqdm(avi_paths, desc="Frame extraction", unit="video"):
        relative = avi_path.relative_to(raw_root).with_suffix("")
        stats = extract_video_frames(avi_path, frames_root / relative, resize)
        total_created += int(stats["created"])
    return total_created


def compute_flow_for_selected(
    avi_paths: list[Path],
    raw_root: Path,
    frames_root: Path,
    flow_root: Path,
) -> int:
    total_created = 0
    for avi_path in tqdm(avi_paths, desc="Optical flow", unit="video"):
        relative = avi_path.relative_to(raw_root).with_suffix("")
        stats = compute_video_flow(frames_root / relative, flow_root / relative)
        total_created += int(stats["created"])
    return total_created


def first_available_video_dir(prepared_root: Path, class_name: str) -> Path:
    class_root = prepared_root / class_name
    video_dirs = sorted(path for path in class_root.iterdir() if path.is_dir()) if class_root.exists() else []
    if not video_dirs:
        raise FileNotFoundError(f"No prepared videos found for class {class_name!r} under {prepared_root}")
    return video_dirs[0]


def evaluate_model_inline(
    model: torch.nn.Module,
    data_loader: DataLoader,
    class_names: tuple[str, ...],
    device: str,
    output_dir: Path,
    mode: str,
) -> dict[str, object]:
    model.eval()
    all_preds: list[int] = []
    all_labels: list[int] = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            logits = model(inputs.to(device))
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())

    accuracy = compute_accuracy(all_preds, all_labels)
    confusion = compute_confusion_matrix(all_preds, all_labels, class_names)
    report = classification_report(all_preds, all_labels, class_names)
    fig = plot_confusion_matrix(confusion, class_names, output_dir / f"confusion_matrix_{mode}.png")
    display(fig)
    plt.close(fig)

    metrics = {
        "mode": mode,
        "accuracy": accuracy,
        "macro_f1": float(report["macro avg"]["f1-score"]),
        "weighted_f1": float(report["weighted avg"]["f1-score"]),
        "confusion_matrix": confusion.tolist(),
        "classification_report": report,
    }
    with (output_dir / f"metrics_{mode}.json").open("w", encoding="utf-8") as handle:
        json.dump(metrics, handle, indent=2)

    print(f"Test accuracy: {accuracy:.4f} | Macro F1: {metrics['macro_f1']:.4f}")
    display(pd.DataFrame(report).transpose().round(3))
    return metrics


print("Notebook helpers ready.")

## 3. Dataset Acquisition

The raw KTH videos live under `data/kth/raw/{class}`. If the dataset is already present, the download step becomes a no-op.

After confirming the raw data, the notebook picks the exact videos it will prepare:

- GPU: full official KTH split.
- CPU: a smaller person-balanced subset per class using the same train/test protocol.

In [ ]:
seed_everything(config.seed)
raw_root = ensure_kth_downloaded(config)
frames_root = config.data_root / "frames"
flow_root = config.data_root / "flow"

selected_avis, selection_summary = select_videos_for_notebook(
    raw_root,
    config.class_names,
    train_per_class=config.train_videos_per_class,
    test_per_class=config.test_videos_per_class,
)

print(f"Raw dataset: {raw_root}")
raw_summary = summarize_raw_videos(raw_root, config.class_names)
raw_summary["total_videos"] = raw_summary["avi_files"].cumsum()
display(raw_summary)
print(f"Total videos available: {raw_summary['avi_files'].sum()}")

display(
    Markdown(
        f"**Notebook subset:** {len(selected_avis)} videos prepared "
        f"({config.train_videos_per_class or 'all'} train per class, "
        f"{config.test_videos_per_class or 'all'} test per class)."
    )
)
display(selection_summary)

## 4. Preprocessing: Frame Extraction

The notebook extracts PNG frames only for the selected videos. Outputs are written under `data/kth/frames/{class}/{video}/frame_XXXXX.png` using the configured resize value.

This keeps the CPU path practical while preserving the same person-based split logic as the full pipeline.

In [ ]:
t0 = time.time()
new_frames = extract_frames_for_selected(selected_avis, raw_root, frames_root, config.resize)
elapsed = time.time() - t0
print(f"Prepared frames for {len(selected_avis)} videos in {elapsed:.1f}s ({new_frames} new PNG files written).")

fig, axes = plt.subplots(len(NOTEBOOK_SAMPLE_CLASSES), 4, figsize=(16, 3 * len(NOTEBOOK_SAMPLE_CLASSES)))
for row, class_name in enumerate(NOTEBOOK_SAMPLE_CLASSES):
    video_dir = first_available_video_dir(frames_root, class_name)
    frame_paths = sorted(video_dir.glob("frame_*.png"))
    step = max(len(frame_paths) // 4, 1)
    for col in range(4):
        index = min(col * step, len(frame_paths) - 1)
        frame = cv2.cvtColor(cv2.imread(str(frame_paths[index])), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(frame)
        axes[row, col].set_title(f"{class_name} / {frame_paths[index].name}", fontsize=9)
        axes[row, col].axis("off")

fig.suptitle("Sample extracted frames across action classes", fontsize=13)
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Preprocessing: Dense Optical Flow

For every consecutive frame pair in the selected subset, the notebook computes dense Farneback optical flow and stores it as `flow_XXXXX.npy` with shape `(H, W, 2)`.

This is still the most expensive preprocessing step, but reruns stay fast because cached flow files are skipped.

In [ ]:
t0 = time.time()
new_flow_files = compute_flow_for_selected(selected_avis, raw_root, frames_root, flow_root)
elapsed = time.time() - t0
print(f"Prepared optical flow for {len(selected_avis)} videos in {elapsed:.1f}s ({new_flow_files} new .npy files written).")

fig, axes = plt.subplots(len(NOTEBOOK_SAMPLE_FLOW_CLASSES), 3, figsize=(14, 4 * len(NOTEBOOK_SAMPLE_FLOW_CLASSES)))
for row, class_name in enumerate(NOTEBOOK_SAMPLE_FLOW_CLASSES):
    video_dir = first_available_video_dir(frames_root, class_name)
    flow_dir = flow_root / class_name / video_dir.name
    frame_paths = sorted(video_dir.glob("frame_*.png"))
    flow_paths = sorted(flow_dir.glob("flow_*.npy"))
    midpoint = len(flow_paths) // 2

    frame_a = cv2.cvtColor(cv2.imread(str(frame_paths[midpoint])), cv2.COLOR_BGR2RGB)
    frame_b = cv2.cvtColor(cv2.imread(str(frame_paths[midpoint + 1])), cv2.COLOR_BGR2RGB)
    flow_hsv = visualize_flow(np.load(flow_paths[midpoint]))

    for col, (image, title) in enumerate(
        zip(
            [frame_a, frame_b, flow_hsv],
            [f"{class_name}: frame t", f"{class_name}: frame t+1", f"{class_name}: optical flow (HSV)"],
            strict=True,
        )
    ):
        axes[row, col].imshow(image)
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis("off")

fig.suptitle("Dense Farneback optical flow samples", fontsize=13)
fig.tight_layout()
display(fig)
plt.close(fig)

## 6. Dataset Split Summary

The KTH protocol uses a person-based split: persons 1-16 for training and persons 17-25 for testing. The notebook keeps that protocol intact even when it uses a smaller CPU subset.

Below we summarize the prepared frame and flow directories that will feed the two experiments.

In [ ]:
flow_train_dirs, flow_train_labels, flow_test_dirs, flow_test_labels = get_train_test_split(config.data_root, mode="flow")
frame_train_dirs, frame_train_labels, frame_test_dirs, frame_test_labels = get_train_test_split(config.data_root, mode="rgb")

if not flow_train_dirs or not flow_test_dirs:
    raise RuntimeError("Flow split is empty. Check preprocessing output under data/kth/flow.")
if not frame_train_dirs or not frame_test_dirs:
    raise RuntimeError("RGB split is empty. Check preprocessing output under data/kth/frames.")


def split_summary(train_labels: list[int], test_labels: list[int], class_names: tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for label, class_name in enumerate(class_names):
        rows.append(
            {
                "class": class_name,
                "train": sum(1 for item in train_labels if item == label),
                "test": sum(1 for item in test_labels if item == label),
            }
        )
    return pd.DataFrame(rows)


print("=== Flow dataset split ===")
display(split_summary(flow_train_labels, flow_test_labels, config.class_names))
print(f"Flow: {len(flow_train_dirs)} train / {len(flow_test_dirs)} test videos")

print("\n=== RGB dataset split ===")
display(split_summary(frame_train_labels, frame_test_labels, config.class_names))
print(f"RGB: {len(frame_train_dirs)} train / {len(frame_test_dirs)} test videos")

---

## 7. Experiment 1: Optical Flow Model

### Architecture

The flow model is a **ResNet-18** with a modified first convolution layer. Instead of 3 RGB channels, it accepts **20 input channels** (10 flow fields × 2 displacement axes). The pretrained ImageNet conv1 weights are inflated by round-robin replication with energy-preserving scaling:

$$W_{\text{new}}[:, c, :, :] = W_{\text{rgb}}[:, c \bmod 3, :, :] \times \frac{3}{C_{\text{in}}}$$

This initialization lets the flow model start from a reasonable feature space rather than random weights, even though the input modality differs from ImageNet images.

### Hypothesis

Optical flow explicitly encodes **motion**, which is the primary discriminative signal for action recognition. We expect the flow model to outperform the RGB baseline, especially on classes with distinctive movement patterns (e.g., boxing vs walking).

In [ ]:
# Experiment 1: Optical Flow
seed_everything(config.seed)

config.mode = "flow"
config.input_channels = config.clip_length * 2
config.output_dir = Path("outputs/flow")
config.output_dir.mkdir(parents=True, exist_ok=True)

flow_train_ds = FlowClipDataset(flow_train_dirs, flow_train_labels, config, train=True)
flow_test_ds = FlowClipDataset(flow_test_dirs, flow_test_labels, config, train=False)

flow_train_loader = DataLoader(flow_train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers)
flow_test_loader = DataLoader(flow_test_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers)

sample, label = flow_train_ds[0]
print(f"Flow input tensor shape: {tuple(sample.shape)} (2 x clip_length channels x H x W)")
print(f"Value range: [{sample.min():.3f}, {sample.max():.3f}]")
print(f"Train: {len(flow_train_ds)} clips | Test: {len(flow_test_ds)} clips")

flow_model = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=config.input_channels,
    pretrained=config.pretrained_backbone,
)
total_params = sum(parameter.numel() for parameter in flow_model.parameters())
print(f"Flow model parameters: {total_params:,}")

### Training

In [ ]:
flow_trainer = Trainer(
    model=flow_model,
    train_loader=flow_train_loader,
    val_loader=flow_test_loader,
    config=config,
    device=config.device,
)
flow_history = flow_trainer.train()

fig = plot_training_curves(flow_history, save_path=config.output_dir / "training_curves_flow.png")
fig.suptitle("Experiment 1: Optical Flow - Training Curves", fontsize=12)
fig.tight_layout()
display(fig)
plt.close(fig)

### Evaluation: Confusion Matrix & Per-Class Metrics

In [ ]:
# Reload best flow checkpoint for evaluation
flow_ckpt_path = config.output_dir / 'best_flow.pt'
flow_ckpt = torch.load(flow_ckpt_path, map_location=config.device, weights_only=False)

flow_best = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=config.input_channels,
    pretrained=False,
)
flow_best.load_state_dict(flow_ckpt['model_state_dict'])
flow_best.to(config.device)

flow_metrics = evaluate_model_inline(
    model=flow_best,
    data_loader=flow_test_loader,
    class_names=config.class_names,
    device=config.device,
    output_dir=config.output_dir,
    mode='flow',
)
print(f'\nBest flow checkpoint saved at epoch {flow_ckpt["epoch"]}')

---

## 8. Experiment 2: RGB Baseline

### Architecture

The RGB model is a standard ResNet-18 with a replaced final FC layer for 6-class classification. Each input is the center frame of a temporal clip, normalized with ImageNet statistics.

### Hypothesis

A single RGB frame captures appearance but not explicit motion. This baseline tests whether static appearance alone is sufficient for KTH action recognition, or whether temporal motion information provides a meaningful advantage.

In [ ]:
# Experiment 2: RGB Baseline
seed_everything(config.seed)

config.mode = "rgb"
config.input_channels = 3
config.output_dir = Path("outputs/rgb")
config.output_dir.mkdir(parents=True, exist_ok=True)

rgb_train_ds = RGBClipDataset(frame_train_dirs, frame_train_labels, config, train=True)
rgb_test_ds = RGBClipDataset(frame_test_dirs, frame_test_labels, config, train=False)

rgb_train_loader = DataLoader(rgb_train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers)
rgb_test_loader = DataLoader(rgb_test_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers)

sample, label = rgb_train_ds[0]
print(f"RGB input tensor shape: {tuple(sample.shape)} (3 channels x H x W)")
print(f"Value range: [{sample.min():.3f}, {sample.max():.3f}]")
print(f"Train: {len(rgb_train_ds)} clips | Test: {len(rgb_test_ds)} clips")

rgb_model = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=3,
    pretrained=config.pretrained_backbone,
)
total_params = sum(parameter.numel() for parameter in rgb_model.parameters())
print(f"RGB model parameters: {total_params:,}")

### Training

In [ ]:
rgb_trainer = Trainer(
    model=rgb_model,
    train_loader=rgb_train_loader,
    val_loader=rgb_test_loader,
    config=config,
    device=config.device,
)
rgb_history = rgb_trainer.train()

fig = plot_training_curves(rgb_history, save_path=config.output_dir / "training_curves_rgb.png")
fig.suptitle("Experiment 2: RGB Baseline - Training Curves", fontsize=12)
fig.tight_layout()
display(fig)
plt.close(fig)

### Evaluation: Confusion Matrix & Per-Class Metrics

In [ ]:
# Reload best RGB checkpoint for evaluation
rgb_ckpt_path = config.output_dir / 'best_rgb.pt'
rgb_ckpt = torch.load(rgb_ckpt_path, map_location=config.device, weights_only=False)

rgb_best = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=3,
    pretrained=False,
)
rgb_best.load_state_dict(rgb_ckpt['model_state_dict'])
rgb_best.to(config.device)

rgb_metrics = evaluate_model_inline(
    model=rgb_best,
    data_loader=rgb_test_loader,
    class_names=config.class_names,
    device=config.device,
    output_dir=config.output_dir,
    mode='rgb',
)
print(f'\nBest RGB checkpoint saved at epoch {rgb_ckpt["epoch"]}')

---

## 9. Comparison: Optical Flow vs RGB

This section brings together the results from both experiments for direct comparison. We compare:

1. **Overall accuracy and macro F1** — the headline numbers.
2. **Per-class F1 scores** — where each model excels or struggles.
3. **Confusion matrices side by side** — which class pairs cause the most confusion.
4. **Training dynamics** — convergence speed and overfitting behaviour.

In [ ]:
# Summary Table
comparison = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Macro F1", "Weighted F1"],
        "Optical Flow": [
            f"{flow_metrics['accuracy']:.4f}",
            f"{flow_metrics['macro_f1']:.4f}",
            f"{flow_metrics['weighted_f1']:.4f}",
        ],
        "RGB Baseline": [
            f"{rgb_metrics['accuracy']:.4f}",
            f"{rgb_metrics['macro_f1']:.4f}",
            f"{rgb_metrics['weighted_f1']:.4f}",
        ],
    }
)
print("=== Experiment Comparison ===")
display(comparison.style.set_caption("Flow vs RGB - Overall Metrics"))

per_class = pd.DataFrame(
    {
        "Class": list(config.class_names),
        "Flow F1": [flow_metrics["classification_report"][class_name]["f1-score"] for class_name in config.class_names],
        "RGB F1": [rgb_metrics["classification_report"][class_name]["f1-score"] for class_name in config.class_names],
    }
)
per_class["Diff (Flow - RGB)"] = per_class["Flow F1"] - per_class["RGB F1"]
per_class = per_class.round(3)
print("\n=== Per-Class F1 Scores ===")
display(per_class.style.set_caption("Per-class F1: positive diff means flow wins"))

x = np.arange(len(config.class_names))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, per_class["Flow F1"], width, label="Optical Flow", color="#4C72B0")
ax.bar(x + width / 2, per_class["RGB F1"], width, label="RGB Baseline", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(config.class_names, rotation=30, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1: Optical Flow vs RGB")
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# ── Side-by-side confusion matrices ──────────────────────────────────────────
flow_cm = np.array(flow_metrics['confusion_matrix'])
rgb_cm = np.array(rgb_metrics['confusion_matrix'])

import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(flow_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=config.class_names, yticklabels=config.class_names, ax=ax1)
ax1.set_title('Optical Flow')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')

sns.heatmap(rgb_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=config.class_names, yticklabels=config.class_names, ax=ax2)
ax2.set_title('RGB Baseline')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')

fig.suptitle('Confusion Matrices: Flow vs RGB', fontsize=13)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# ── Training dynamics comparison ──────────────────────────────────────────────
epochs_range = range(1, config.epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, flow_history['val_loss'], label='Flow val loss', color='#4C72B0')
axes[0].plot(epochs_range, rgb_history['val_loss'], label='RGB val loss', color='#DD8452')
axes[0].plot(epochs_range, flow_history['train_loss'], '--', alpha=0.5, label='Flow train loss', color='#4C72B0')
axes[0].plot(epochs_range, rgb_history['train_loss'], '--', alpha=0.5, label='RGB train loss', color='#DD8452')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)

axes[1].plot(epochs_range, flow_history['val_acc'], label='Flow val acc', color='#4C72B0')
axes[1].plot(epochs_range, rgb_history['val_acc'], label='RGB val acc', color='#DD8452')
axes[1].plot(epochs_range, flow_history['train_acc'], '--', alpha=0.5, label='Flow train acc', color='#4C72B0')
axes[1].plot(epochs_range, rgb_history['train_acc'], '--', alpha=0.5, label='RGB train acc', color='#DD8452')
axes[1].set_title('Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)

fig.suptitle('Training Dynamics: Flow vs RGB', fontsize=12)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# ── Save combined comparison JSON ─────────────────────────────────────────────
comparison_payload = {
    'flow': {
        'accuracy': flow_metrics['accuracy'],
        'macro_f1': flow_metrics['macro_f1'],
        'weighted_f1': flow_metrics['weighted_f1'],
    },
    'rgb': {
        'accuracy': rgb_metrics['accuracy'],
        'macro_f1': rgb_metrics['macro_f1'],
        'weighted_f1': rgb_metrics['weighted_f1'],
    },
    'per_class_f1': {
        cn: {
            'flow': flow_metrics['classification_report'][cn]['f1-score'],
            'rgb': rgb_metrics['classification_report'][cn]['f1-score'],
        }
        for cn in config.class_names
    },
}

Path('outputs').mkdir(exist_ok=True)
with open('outputs/comparison.json', 'w') as f:
    json.dump(comparison_payload, f, indent=2)
print('Saved outputs/comparison.json')

# ── Print saved artifacts ────────────────────────────────────────────────────
print('\n=== Saved Artifacts ===')
for d in [Path('outputs/flow'), Path('outputs/rgb')]:
    for p in sorted(d.iterdir()):
        print(f'  {p}')

---

## 10. Conclusion

This notebook runs two complete experiments on the KTH Actions dataset and writes the key artifacts below:

| Artifact | Flow | RGB |
|----------|------|-----|
| Best checkpoint | `outputs/flow/best_flow.pt` | `outputs/rgb/best_rgb.pt` |
| Training curves | `outputs/flow/training_curves_flow.png` | `outputs/rgb/training_curves_rgb.png` |
| Confusion matrix | `outputs/flow/confusion_matrix_flow.png` | `outputs/rgb/confusion_matrix_rgb.png` |
| Metrics JSON | `outputs/flow/metrics_flow.json` | `outputs/rgb/metrics_rgb.json` |
| Comparison | `outputs/comparison.json` | |

CPU runs use a smaller person-balanced subset by default; GPU runs use the full dataset. In both cases, preprocessing stays cache-aware, so reruns skip work that has already been completed.